In [ ]:
import os

# ----------------------- Azure OpenAI Configuration -----------------------
# Replace these with your actual values
AZURE_FOUNDRY_API_KEY = "YOUR_AZURE_FOUNDRY_KEY"
AZURE_FOUNDRY_ENDPOINT = "https://sustainability-resource.openai.azure.com/"
AZURE_FOUNDRY_API_VERSION = "2024-02-15-preview"
AZURE_FOUNDRY_CHAT_DEPLOYMENT = "grok-4-fast-non-reasoning"  

# Set as environment variables
os.environ["AZURE_FOUNDRY_API_KEY"] = AZURE_FOUNDRY_API_KEY
os.environ["AZURE_FOUNDRY_ENDPOINT"] = AZURE_FOUNDRY_ENDPOINT
os.environ["AZURE_FOUNDRY_API_VERSION"] = AZURE_FOUNDRY_API_VERSION
os.environ["AZURE_FOUNDRY_CHAT_DEPLOYMENT"] = AZURE_FOUNDRY_CHAT_DEPLOYMENT

print("✅ Configuration loaded successfully!")
print(f"Endpoint: {AZURE_FOUNDRY_ENDPOINT}")
print(f"Deployment: {AZURE_FOUNDRY_CHAT_DEPLOYMENT}")
print(f"API Version: {AZURE_FOUNDRY_API_VERSION}")

✅ Configuration loaded successfully!
Endpoint: https://sustainability-resource.openai.azure.com/
Deployment: grok-4-fast-non-reasoning
API Version: 2024-02-15-preview


In [ ]:
import pandas as pd
import os
import re
import time
import json
from openai import AzureOpenAI

# ----------------------- Load Configuration -----------------------
AZURE_FOUNDRY_API_KEY = os.environ.get("AZURE_FOUNDRY_API_KEY")
AZURE_FOUNDRY_ENDPOINT = os.environ.get("AZURE_FOUNDRY_ENDPOINT")
AZURE_FOUNDRY_API_VERSION = os.environ.get("AZURE_FOUNDRY_API_VERSION", "2024-02-15-preview")
AZURE_FOUNDRY_CHAT_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_CHAT_DEPLOYMENT", "DeepSeek-V3.2")

if not AZURE_FOUNDRY_API_KEY or not AZURE_FOUNDRY_ENDPOINT:
    raise ValueError(" Please run Cell 1 first to set AZURE_FOUNDRY_API_KEY and AZURE_FOUNDRY_ENDPOINT.")

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=AZURE_FOUNDRY_API_KEY,
    api_version=AZURE_FOUNDRY_API_VERSION,
    azure_endpoint=AZURE_FOUNDRY_ENDPOINT,
)

print("✅ Azure OpenAI client initialized successfully!")

MAX_RETRIES = 3

# Output file configuration
OUTPUT_DIRECTORY = "/path/to/your/output/folder"  # Change this to your desired output path

# ----------------------- USER MESSAGE TEMPLATE (What the model sees as input) -----------------------
USER_MESSAGE_TEMPLATE = """You are an expert UN Sustainable Development Goals (SDG) classifier.

**THE 17 UN SDGs:**
SDG1: No Poverty | SDG2: Zero Hunger | SDG3: Good Health and Well-Being | SDG4: Quality Education
SDG5: Gender Equality | SDG6: Clean Water and Sanitation | SDG7: Affordable and Clean Energy
SDG8: Decent Work and Economic Growth | SDG9: Industry, Innovation, Infrastructure | SDG10: Reduced Inequalities
SDG11: Sustainable Cities and Communities | SDG12: Responsible Consumption and Production | SDG13: Climate Action
SDG14: Life Below Water | SDG15: Life on Land | SDG16: Peace, Justice, Strong Institutions
SDG17: Partnerships for the Goals

Analyze the following text and classify it according to the UN SDGs. Provide your reasoning step-by-step:

**Step 1: Understand the paragraph** - Identify the main subject, key entities, and explicit actions/outcomes.

**Step 2: Identify potential SDG connections** - Extract keywords/phrases and map them to potential SDGs with textual evidence.

**Step 3: Final analysis** - Explain which SDG(s) apply based on direct textual evidence, or explain why it's NOSDG. Address why other potential SDGs were excluded.

**Conclusion** - One sentence stating the final classification.

**Text to analyze:**
{text}"""

# ----------------------- SYSTEM PROMPT FOR GENERATION -----------------------
GENERATION_SYSTEM_PROMPT = """You are an expert UN Sustainable Development Goals (SDG) Analyst.
I will provide you with a text and a **VERIFIED LABEL**.

Your task is to generate a step-by-step "Chain of Thought" that logically derives this specific label from the text.

**THE 17 UN SDGs FOR YOUR REFERENCE:**
SDG1: No Poverty | SDG2: Zero Hunger | SDG3: Good Health and Well-Being | SDG4: Quality Education
SDG5: Gender Equality | SDG6: Clean Water and Sanitation | SDG7: Affordable and Clean Energy
SDG8: Decent Work and Economic Growth | SDG9: Industry, Innovation, Infrastructure | SDG10: Reduced Inequalities
SDG11: Sustainable Cities and Communities | SDG12: Responsible Consumption and Production | SDG13: Climate Action
SDG14: Life Below Water | SDG15: Life on Land | SDG16: Peace, Justice, Strong Institutions
SDG17: Partnerships for the Goals

**YOUR REASONING PROCESS:**

* **Step 1: Understand the Paragraph**
    * Identify what is being described: the main subject, key actors or entities mentioned, and the specific actions, programs, or outcomes discussed.
    * Extract the core message without indirect interpretation—focus on what is explicitly stated.

* **Step 2: Identify Potential SDG Connections**
    * Extract idea, meaning, phrases, and concepts from the text.
    * Map these textual elements to SDG themes by asking: which SDG core areas do these words/phrases directly relate to?
    * List 2-4 potential SDGs that could be relevant based on the textual evidence.
    * For each potential SDG, cite the specific words or phrases from the text that suggest this connection.

* **Step 3: Final Analysis & Selection**
    * Compare your potential SDGs against the **VERIFIED LABEL** but do not mention in the output that you are comparing with verified label.
    * Provide clear evidence from the text that confirms why the verified label is correct.
    * If the verified label includes multiple SDGs, explain the evidence for each one.
    * If the verified label is NOSDG, explain why the text lacks sufficient SDG-related content (e.g., purely procedural, financial reporting, or no development/sustainability focus).
    * Address why any other potential SDGs were not included in the final label.

* **Conclusion**
    * One clear sentence stating the final classification and the primary reason it was chosen.

**OUTPUT FORMAT (DO NOT include <think> or <SDG> tags):**

Step 1: Let me Understand the paragraph
[Your analysis]

Step 2: Based on the analysis the possible SDG's are
[Your potential SDG mappings with evidence]

Step 3: From my analysis here are the final SDG classification and Justification
[Your final reasoning]

Conclusion:
[Your conclusion]

Final Classification: [SDG labels only, e.g., "SDG1, SDG8" or "NOSDG"]

---
**EXAMPLES:**

**Example 1:**
Text: "Donations and bequests are recognised as revenue when the right to receive the fund or asset has been established..."
Verified Label: NOSDG

Response:
Step 1: Let me understand the paragraph
The text describes accounting procedures for recognizing donations and bequests as revenue in financial statements. It focuses on the timing and conditions for revenue recognition.

Step 2: Based on the analysis the possible SDG's are
- Potential SDG17: The word "donations" might suggest partnerships, but the text discusses accounting treatment, not partnership activities or development cooperation.
- No other clear SDG keywords present. The text contains financial/accounting terminology ("revenue," "asset," "recognized") without development or sustainability context.

Step 3: From my analysis here are the final SDG classification and Justification
The predicted label is NOSDG because the text is purely procedural and administrative. It describes financial reporting standards without addressing any social, environmental, or economic development outcomes. There are no explicit connections to poverty reduction, health, education, environment, or any other SDG theme. The mention of "donations" alone does not constitute SDG-relevant content when discussed purely in accounting terms.

Conclusion:
The text is classified as NOSDG because it describes technical accounting procedures without any explicit development or sustainability focus.

Final Classification: NOSDG

**Example 2:**
Text: "Part-time work is also further disaggregated into part-time temporary and part-time permanent jobs... Permanent full-time employment remains nonetheless the norm..."
Verified Label: SDG1, SDG8

Response:
Step 1: Let me Understand the paragraph
The text analyzes employment data, breaking down work into categories: part-time vs. full-time, and temporary vs. permanent. It notes that permanent full-time employment is still the predominant form.

Step 2: Based on the analysis the possible SDG are
- Potential SDG8: "employment," "jobs," "work," "permanent," "full-time" all directly relate to decent work and economic growth themes.
- Potential SDG1: "permanent" vs "temporary" employment relates to job security and income stability, which are key factors in economic vulnerability and poverty.
- Potential SDG10: Employment quality could relate to inequality, but the text doesn't explicitly discuss inequality or disparities between groups.

Step 3: From my analysis here are the final SDG classification and Justification
The predicted label includes SDG8 and SDG1. For SDG8, the explicit focus on "employment," "jobs," and work categories (full-time, part-time, permanent, temporary) directly addresses employment quality and job characteristics, which are core to decent work. For SDG1, the distinction between "permanent" and "temporary" jobs relates to economic security—permanent employment provides stable income while temporary work creates economic vulnerability, connecting to poverty dimensions. SDG10 was excluded because the text analyzes employment types generally without explicitly addressing inequality between population groups.

Conclusion:
The text is classified as SDG1 and SDG8 because it explicitly analyzes employment structures and job security, directly relating to economic stability and decent work.

Final Classification: SDG1, SDG8

**Example 3:**
Text: "The 1.5C target was a critical negotiating issue for the SIDS, as many of the climate impacts anticipated even at a 2C global mean temperature rise are very likely to result in catastrophic damage to these states."
Verified Label: SDG3

Response:
Step 1: Let me understand the paragraph
The text discusses negotiations involving Small Island Developing States (SIDS) regarding temperature targets (1.5C vs 2C). It emphasizes that temperature increases would cause catastrophic damage to these states.

Step 2: Based on the analysis the possible SDG are
- Potential SDG13: "climate impacts," "temperature rise," "1.5C target," "2C global mean temperature" all explicitly relate to climate change and climate action.
- Potential SDG3: "catastrophic damage to these states" implies severe threats to populations, including health impacts, injuries, and loss of life from climate disasters.
- Potential SDG11: SIDS are specific geographic entities/communities, and "damage to these states" could relate to sustainable cities and communities.

Step 3: From my analysis here are the final SDG classification and Justification
The predicted label is SDG3. While SDG13 seems strongly relevant due to explicit climate terminology, the label focuses on SDG3 because "catastrophic damage" emphasizes the human health and well-being consequences of climate change. The phrase "catastrophic damage to these states" implies severe health risks, mortality, injury, and threats to human survival from climate-related disasters (floods, storms, heat). The classification prioritizes the humanitarian and health impact dimension over the climate mechanism itself. SDG13 was likely excluded because the focus is on the consequence (health/survival) rather than the climate action or mitigation aspect.

Conclusion:
The text is classified as SDG3 because it emphasizes catastrophic human impacts and threats to population health and survival resulting from climate change.

Final Classification: SDG3

"""

GENERATION_USER_PROMPT = """**Text to analyze:**
{text}

**Verified Label:** {label}

Generate the step-by-step reasoning that leads to this verified label."""

# ----------------------- LLM Chain of Thought Generation -----------------------
def generate_cot_reasoning(text, verified_label, retries=MAX_RETRIES):
    """
    Generate Chain of Thought reasoning for SFT fine-tuning.
    Returns the clean assistant response without tags.
    """
    for attempt in range(retries):
        try:
            api_params = {
                "model": AZURE_FOUNDRY_CHAT_DEPLOYMENT,
                "messages": [
                    {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
                    {"role": "user", "content": GENERATION_USER_PROMPT.format(text=text, label=verified_label)}
                ]
            }
            
            response = client.chat.completions.create(**api_params)
            assistant_response = response.choices[0].message.content

            if not assistant_response or assistant_response.strip() == "":
                return "Model returned empty content", verified_label, False

            assistant_response = assistant_response.strip()
            
            # Verify the response contains the expected structure
            has_step1 = "Step 1:" in assistant_response
            has_step2 = "Step 2:" in assistant_response
            has_step3 = "Step 3:" in assistant_response
            has_conclusion = "Conclusion:" in assistant_response
            has_classification = "Final Classification:" in assistant_response
            
            if not all([has_step1, has_step2, has_step3, has_conclusion, has_classification]):
                print(f"⚠️ Warning: Response missing expected structure")
                return assistant_response, verified_label, False

            # Extract the final classification to verify it matches
            classification_match = re.search(r'Final Classification:\s*(.+?)(?:\n|$)', assistant_response, re.IGNORECASE)
            extracted_classification = classification_match.group(1).strip() if classification_match else ""

            return assistant_response, extracted_classification, True

        except Exception as e:
            print(f"❌ Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"CoT Generation Failed: {str(e)}", verified_label, False

    return "Max retries exceeded", verified_label, False

# ----------------------- Excel Processing with Multiple Output Formats -----------------------
def process_excel_for_sft(input_excel_path, output_dir=None):
    """
    Process Excel file and generate outputs in multiple formats:
    1. JSONL for direct SFT training (OpenAI/HuggingFace format)
    2. Excel with all data for review
    3. Alpaca-style JSONL (alternative format)
    """
    if output_dir is None:
        output_dir = OUTPUT_DIRECTORY

    os.makedirs(output_dir, exist_ok=True)

    input_filename = os.path.basename(input_excel_path)
    base_name = os.path.splitext(input_filename)[0]
    
    # Output file paths
    output_excel_path = os.path.join(output_dir, f"{base_name}_SFT_Training_Data.xlsx")
    output_jsonl_path = os.path.join(output_dir, f"{base_name}_SFT_ChatML.jsonl")
    output_alpaca_path = os.path.join(output_dir, f"{base_name}_SFT_Alpaca.jsonl")

    print(f"📂 Input file: {input_excel_path}")
    print(f"📂 Output Excel: {output_excel_path}")
    print(f"📂 Output JSONL (ChatML): {output_jsonl_path}")
    print(f"📂 Output JSONL (Alpaca): {output_alpaca_path}")
    print(f"🤖 Model: {AZURE_FOUNDRY_CHAT_DEPLOYMENT}")
    print("="*80)

    # Initialize JSONL files (clear any existing data)
    try:
        with open(output_jsonl_path, 'w', encoding='utf-8') as f:
            pass  # Create/clear the file
        with open(output_alpaca_path, 'w', encoding='utf-8') as f:
            pass  # Create/clear the file
    except Exception as e:
        print(f"❌ Error initializing JSONL files: {e}")
        return None

    # Read the Excel file
    try:
        df = pd.read_excel(input_excel_path)
    except Exception as e:
        print(f"❌ Error reading Excel file: {e}")
        return None

    # Validate required columns
    required_columns = ['uid', 'text', 'sdg']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"❌ Missing required columns: {missing_columns}")
        return None

    # Prepare output columns for Excel
    df['assistant_response'] = ""
    df['extracted_classification'] = ""
    df['generation_success'] = False

    total_rows = len(df)
    successful_generations = 0

    # Process each row
    for idx, row in df.iterrows():
        uid = row['uid']
        text = str(row['text']).strip() if pd.notna(row['text']) else ""
        verified_label = str(row['sdg']).strip() if pd.notna(row['sdg']) else "NOSDG"

        print(f"\n📝 Processing row {idx + 1}/{total_rows}")
        print(f"   UID: {uid}")
        print(f"   Verified Label: {verified_label}")
        print(f"   Text preview: {text[:100]}..." if len(text) > 100 else f"   Text: {text}")

        if not text:
            df.at[idx, 'assistant_response'] = "Empty text - no response generated"
            df.at[idx, 'extracted_classification'] = "N/A"
            df.at[idx, 'generation_success'] = False
        else:
            # Generate Chain of Thought reasoning
            assistant_response, extracted_classification, success = generate_cot_reasoning(text, verified_label)

            if success:
                successful_generations += 1
                print(f"   ✓ Generation successful")
                print(f"   ✓ Extracted: {extracted_classification}")
            else:
                print(f"   ⚠️ Generation had issues")

            # Store in DataFrame
            df.at[idx, 'assistant_response'] = assistant_response
            df.at[idx, 'extracted_classification'] = extracted_classification
            df.at[idx, 'generation_success'] = success

            # ChatML format (OpenAI/HuggingFace standard) with system message
            chatml_example = {
                "messages": [
                    {"role": "system", "content": "You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification."},
                    {"role": "user", "content": text},
                    {"role": "assistant", "content": assistant_response}
                ]
            }

            # Alpaca format (alternative)
            alpaca_example = {
                "instruction": "You are an expert UN Sustainable Development Goals (SDG) classifier. Analyze the text and classify it according to the UN SDGs. Provide step-by-step reasoning.",
                "input": text,
                "output": assistant_response
            }

            # Save JSONL files in real-time (append mode)
            try:
                with open(output_jsonl_path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(chatml_example, ensure_ascii=False) + '\n')
                
                with open(output_alpaca_path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(alpaca_example, ensure_ascii=False) + '\n')
            except Exception as e:
                print(f"   ⚠️ Error saving JSONL: {e}")

        # Save Excel file in real-time after each row
        try:
            df.to_excel(output_excel_path, index=False, engine='openpyxl')
            print(f"   💾 Progress saved")
        except Exception as e:
            print(f"   ⚠️ Error saving Excel: {e}")

    # Save Excel file one final time
    try:
        df.to_excel(output_excel_path, index=False, engine='openpyxl')
        print(f"✅ Final Excel file saved!")
    except Exception as e:
        print(f"❌ Error saving final Excel file: {e}")

    print("\n" + "="*80)
    print(f"📊 Statistics:")
    print(f"   Total rows processed: {total_rows}")
    print(f"   Successful generations: {successful_generations}")
    print(f"   Success rate: {(successful_generations/total_rows*100):.1f}%")
    print(f"\n📄 Output files:")
    print(f"   Excel (review): {output_excel_path}")
    print(f"   JSONL (ChatML - for OpenAI/HF): {output_jsonl_path}")
    print(f"   JSONL (Alpaca - alternative): {output_alpaca_path}")
    
    return {
        'excel': output_excel_path,
        'chatml': output_jsonl_path,
        'alpaca': output_alpaca_path
    }

# ----------------------- Run -----------------------
# Configure your paths here
input_excel_path = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\input_files\split_2_final_classification.xlsx"
output_directory = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\results\split_2_result_finetune"

# Run the SFT dataset generation
output_files = process_excel_for_sft(input_excel_path, output_directory)
if output_files:
    print(f"\n✅ All outputs saved successfully!")
    print(f"\n💡 Next steps:")
    print(f"   1. Review the Excel file to verify quality")
    print(f"   2. Use the ChatML JSONL for OpenAI fine-tuning or HuggingFace training")
    print(f"   3. Or use the Alpaca JSONL for Alpaca-style training")
else:
    print("\n❌ Processing failed. Please check the errors above.")

✅ Azure OpenAI client initialized successfully!
📂 Input file: D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\input_files\split_2_final_classification.xlsx
📂 Output Excel: D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\results\split_2_result_finetune\split_2_final_classification_SFT_Training_Data.xlsx
📂 Output JSONL (ChatML): D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\results\split_2_result_finetune\split_2_final_classification_SFT_ChatML.jsonl
📂 Output JSONL (Alpaca): D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\fine_tune_data_prep\results\split_2_result_finetune\split_2_final_classification_SFT_Alpaca.jsonl
🤖 Model: grok-4-fast-non-reasoning

📝 Processing row 1/1799
   UID: NOSDG_101
   Verified Label: NOSDG
   Text preview: Number of surveyed Penn employees living here (n=4111) University of Pennsylvania 300 200 100 Philad...
   ✓ Generation successful
   ✓ Extracted: NOSDG
   💾 Progress sav